# EMS MVP — Guia Técnico Completo de Implantação em AWS EKS

> **Objetivo:** servir como material técnico de referência para desenvolvedores e engenheiros para reproduzir, operar e remover a implantação da aplicação EMS MVP em Amazon EKS.

---

## Como ler este notebook

Este documento foi estruturado como um notebook técnico.  
Cada seção procura responder quatro perguntas:

1. **O que é este componente?**
2. **Por que ele existe na arquitetura?**
3. **Como configurá-lo?**
4. **Como validar e diagnosticar problemas?**

A intenção é que o leitor não apenas repita comandos, mas compreenda o raciocínio operacional por trás da solução.


# 1. Escopo da Solução

## Contexto técnico

A aplicação **EMS MVP** é uma API Python executada em Flask/Gunicorn e integrada ao **Amazon Bedrock**.  
A implantação alvo utiliza:

- **Docker** para empacotamento da aplicação
- **Amazon ECR** para armazenamento da imagem do container
- **Amazon EKS** como plano de orquestração Kubernetes
- **Managed Node Group** para execução dos pods
- **IRSA (IAM Roles for Service Accounts)** para autenticação segura com serviços AWS
- **AWS Load Balancer** para exposição externa da aplicação

## O que este guia cobre

Este guia cobre a **implantação final funcional**:

- build da imagem
- publicação no ECR
- criação do cluster EKS
- criação do node group final funcional
- associação do OIDC provider
- criação da policy IAM
- criação da ServiceAccount com IRSA
- criação do Deployment e do Service
- validação funcional da API
- troubleshooting dos problemas reais encontrados
- limpeza completa do ambiente

## O que este guia não cobre

Este documento **não** tenta ensinar Kubernetes do zero, mas explica os conceitos necessários para entender as decisões feitas na implantação.


# 2. Arquitetura da Solução

## Contexto técnico

A solução final implementa um fluxo padrão de execução de uma aplicação containerizada em Kubernetes gerenciado pela AWS.

A imagem da aplicação é construída localmente, enviada para o **Amazon ECR** e então consumida por um **Deployment** no cluster EKS.  
O pod roda em um node EC2 do nodegroup gerenciado.  
A autenticação do pod com serviços AWS é feita via **IRSA**, e o acesso externo à API é fornecido por um **Service do tipo LoadBalancer**.

## Diagrama lógico

```text
Desenvolvedor
    │
    ▼
Docker build local
    │
    ▼
Imagem Docker (ems-mvp:latest)
    │
    ▼
Amazon ECR
    │
    ▼
Amazon EKS
    │
    ├── Control Plane (gerenciado pela AWS)
    │
    └── Managed Node Group (EC2)
            │
            ▼
        Kubernetes Pod
            │
            ├── ServiceAccount Kubernetes
            │       │
            │       └── IAM Role via IRSA
            │               │
            │               └── Amazon Bedrock
            │
            ▼
Kubernetes Service (LoadBalancer)
    │
    ▼
AWS Elastic Load Balancer
    │
    ▼
Cliente HTTP / Postman / curl
```

## Componentes e responsabilidades

| Componente | Responsabilidade |
|---|---|
| Docker | Empacotar aplicação e dependências em uma imagem reproduzível |
| ECR | Armazenar e versionar imagens Docker na AWS |
| EKS | Orquestrar pods e serviços Kubernetes |
| Node Group | Disponibilizar capacidade computacional para os pods |
| Deployment | Declarar como a aplicação deve ser executada |
| Service | Expor a aplicação dentro/fora do cluster |
| OIDC Provider | Permitir trust entre Kubernetes e IAM |
| IRSA | Dar permissões AWS ao pod sem access keys |
| Bedrock | Serviço de IA consumido pela aplicação |


# 3. Conceitos Fundamentais

## 3.1 Docker

Docker empacota a aplicação e o ambiente de execução em uma imagem imutável.

Sem Docker, a aplicação depende do ambiente local do desenvolvedor ou do servidor.  
Com Docker, o que roda no notebook, no servidor e no cluster é essencialmente o mesmo artefato.

A imagem da EMS MVP precisa conter:

- Python
- dependências do `requirements.txt`
- código da aplicação
- Gunicorn
- certificado corporativo confiável no trust store do Linux

## 3.2 Kubernetes

Kubernetes é a camada de orquestração de containers.

| Recurso | Função |
|---|---|
| Pod | Menor unidade executável |
| Deployment | Garante que a quantidade desejada de pods exista |
| Service | Expõe pods por um endereço estável |
| Node | Máquina que executa os pods |

## 3.3 Amazon EKS

Amazon EKS é Kubernetes gerenciado pela AWS.

**Control plane** e **nodes** são coisas diferentes:

- **Control plane**: API server, scheduler, controllers, etcd — gerenciados pela AWS
- **Nodes**: instâncias EC2 que executam os pods

## 3.4 Amazon ECR

ECR é o registry de imagens Docker da AWS.

Fluxo:

```text
docker build → docker tag → docker push → EKS faz pull
```

## 3.5 OIDC e IRSA

IRSA = **IAM Roles for Service Accounts**

Fluxo:

```text
Pod
↓
ServiceAccount Kubernetes
↓
Token OIDC
↓
STS AssumeRoleWithWebIdentity
↓
Credenciais temporárias
↓
Serviço AWS
```

Benefícios:

- elimina credenciais estáticas
- reduz risco de vazamento
- segue princípio de menor privilégio
- facilita auditoria e governança


# 4. Pré-requisitos do Ambiente Local

## Contexto técnico

A implantação foi executada a partir de um ambiente **Windows 11 + WSL2**, com as ferramentas Linux instaladas dentro do WSL.

Essa decisão é importante porque ferramentas como `kubectl`, `eksctl`, `aws`, `curl` e `docker` se comportam de maneira mais previsível em shell Linux do que em PowerShell para fluxos DevOps.

## Ferramentas necessárias

| Ferramenta | Função |
|---|---|
| WSL2 | Ambiente Linux local |
| Docker Desktop | Build e execução local de containers |
| AWS CLI v2 | Interação com serviços AWS |
| kubectl | Administração do cluster Kubernetes |
| eksctl | Provisionamento do cluster EKS |

## Verificações iniciais

```bash
aws --version
docker --version
kubectl version --client
eksctl version
```


# 5. Certificado Corporativo e Ambiente com Proxy

## Contexto técnico

Durante a implantação foi identificado um ambiente corporativo com **inspeção TLS**.  
Isso afeta o comportamento de ferramentas como:

- `curl`
- `aws cli`
- `boto3`
- `botocore`
- `requests`

## Sintoma típico

```text
SSL: CERTIFICATE_VERIFY_FAILED
unable to get local issuer certificate
```

## Estratégia adotada

1. Instalar o certificado corporativo no WSL2
2. Incluir o certificado na imagem Docker
3. Atualizar o trust store dentro do container

## Conversão do certificado para PEM/CRT

```bash
mkdir -p certs
cp /mnt/c/Users/mmaccafe/certs/corp-root.cer certs/
openssl x509 -inform DER -in certs/corp-root.cer -out certs/corp-root.crt
```

Adicionar ao `.gitignore`:

```text
certs/corp-root.cer
certs/corp-root.crt
```


# 6. Containerização da Aplicação

## Contexto técnico

A aplicação EMS MVP é uma API Python/Flask, servida em produção por **Gunicorn**.  
Isso é importante porque o servidor de desenvolvimento do Flask (`app.run`) não é adequado para produção nem para ambientes como Kubernetes.

## Responsabilidades do Dockerfile

- definir base Python
- criar diretório de trabalho
- instalar `ca-certificates`
- copiar `corp-root.crt`
- executar `update-ca-certificates`
- instalar dependências do `requirements.txt`
- copiar o projeto
- iniciar Gunicorn

## Variáveis úteis na imagem

```dockerfile
ENV AWS_CA_BUNDLE=/etc/ssl/certs/ca-certificates.crt
ENV REQUESTS_CA_BUNDLE=/etc/ssl/certs/ca-certificates.crt
```

## Build local

```bash
docker build -t ems-mvp:latest .
```

## Teste local

```bash
docker run -p 8080:8080 ems-mvp:latest
```

## Endpoint local de teste

```bash
curl -X POST http://localhost:8080/v1/ai/turn   -H "Content-Type: application/json"   -d '{}'
```


# 7. Credenciais AWS no Desenvolvimento Local

## Contexto técnico

Durante o desenvolvimento local em Docker, a aplicação precisou acessar a AWS para chamar o Bedrock.  
O container local não enxerga automaticamente as credenciais do host.

## Sintoma típico

```text
Unable to locate credentials
```

## Solução usada localmente

Exemplo com montagem de `~/.aws`:

```bash
docker run -d --name ems-mvp   -p 8080:8080   -v /home/mmaccafe/.aws:/root/.aws:ro   -e AWS_REGION=us-east-1   ems-mvp:latest
```

> Observação: essa estratégia foi usada apenas em desenvolvimento local.  
> No EKS, a autenticação correta é via **IRSA**.


# 8. Publicação da Imagem no Amazon ECR

## Contexto técnico

O cluster EKS precisa buscar a imagem de algum registry acessível.  
A imagem local do notebook não é visível para o cluster, então ela precisa ser publicada no ECR.

## Variáveis usadas

```bash
ACCOUNT_ID=746443183845
REGION=us-east-1
REPO_NAME=ems-mvp
IMAGE_TAG=latest
IMAGE_URI=${ACCOUNT_ID}.dkr.ecr.${REGION}.amazonaws.com/${REPO_NAME}:${IMAGE_TAG}
```

## Criar o repositório

```bash
aws ecr create-repository   --repository-name ems-mvp   --region us-east-1
```

## Autenticar Docker no ECR

```bash
aws ecr get-login-password --region us-east-1 | docker login --username AWS --password-stdin 746443183845.dkr.ecr.us-east-1.amazonaws.com
```

## Taguear a imagem

```bash
docker tag ems-mvp:latest 746443183845.dkr.ecr.us-east-1.amazonaws.com/ems-mvp:latest
```

## Enviar a imagem

```bash
docker push 746443183845.dkr.ecr.us-east-1.amazonaws.com/ems-mvp:latest
```


# 9. Criação do Cluster Amazon EKS

## Contexto técnico

O cluster foi criado com `eksctl`, que usa CloudFormation por trás para provisionar:

- cluster EKS
- networking
- security groups
- nodegroups gerenciados

## Comando final funcional

```bash
eksctl create cluster   --name ems-mvp-cluster   --region us-east-1   --nodegroup-name standard-workers-small   --node-type t3.small   --nodes 1   --nodes-min 1   --nodes-max 1   --managed
```

## Validações após criação

```bash
aws eks update-kubeconfig --region us-east-1 --name ems-mvp-cluster
kubectl config current-context
kubectl cluster-info
kubectl get nodes
```

Resultado esperado:

```text
STATUS: Ready
```


# 10. OIDC Provider

## Contexto técnico

OIDC é o componente que estabelece confiança entre o Kubernetes e o IAM.  
Sem ele, a ServiceAccount do Kubernetes não consegue assumir uma role IAM.

## Comando

```bash
eksctl utils associate-iam-oidc-provider   --cluster ems-mvp-cluster   --region us-east-1   --approve
```


# 11. Policy IAM para o Bedrock

## Contexto técnico

A aplicação precisa chamar operações do Amazon Bedrock.  
Para isso, a role assumida pelo pod precisa ter permissão explícita.

## Policy utilizada

Arquivo: `bedrock-policy.json`

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "BedrockInvokeAndRetrieve",
      "Effect": "Allow",
      "Action": [
        "bedrock:InvokeModel",
        "bedrock:InvokeModelWithResponseStream",
        "bedrock:Retrieve"
      ],
      "Resource": "*"
    }
  ]
}
```

## Criar a policy

```bash
aws iam create-policy   --policy-name BedrockAccessPolicy   --policy-document file://bedrock-policy.json
```


# 12. IRSA — ServiceAccount com Role IAM

## Contexto técnico

Esta etapa substitui o uso de access keys no cluster.

## Comando

```bash
eksctl create iamserviceaccount   --cluster ems-mvp-cluster   --region us-east-1   --namespace default   --name ems-mvp-sa   --attach-policy-arn arn:aws:iam::746443183845:policy/BedrockAccessPolicy   --approve
```

## Validação

```bash
kubectl describe serviceaccount ems-mvp-sa
```

Verificar a annotation:

```text
eks.amazonaws.com/role-arn
```


# 13. Manifesto Kubernetes da Aplicação

## Contexto técnico

O Deployment descreve como a aplicação deve rodar.  
O Service descreve como ela será exposta.

## Arquivo `k8s/ems-mvp.yaml`

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: ems-mvp
  namespace: default
spec:
  replicas: 1
  selector:
    matchLabels:
      app: ems-mvp
  template:
    metadata:
      labels:
        app: ems-mvp
    spec:
      serviceAccountName: ems-mvp-sa
      containers:
        - name: ems-mvp
          image: 746443183845.dkr.ecr.us-east-1.amazonaws.com/ems-mvp:latest
          imagePullPolicy: Always
          ports:
            - containerPort: 8080
          env:
            - name: AWS_REGION
              value: "us-east-1"
            - name: BEDROCK_KB_ID
              value: "BJWEDBPNJH"
          resources:
            requests:
              cpu: "200m"
              memory: "256Mi"
            limits:
              cpu: "500m"
              memory: "512Mi"
---
apiVersion: v1
kind: Service
metadata:
  name: ems-mvp-service
  namespace: default
spec:
  selector:
    app: ems-mvp
  ports:
    - protocol: TCP
      port: 80
      targetPort: 8080
  type: LoadBalancer
```

## Aplicar

```bash
kubectl apply -f k8s/ems-mvp.yaml
```


# 14. Aplicação e Validação

## Verificar o pod

```bash
kubectl get pods
kubectl get pods -w
```

## Verificar o service

```bash
kubectl get svc
```

## Endpoint externo

O Service expõe:

- porta **80** externamente
- `targetPort: 8080` internamente

Portanto, o endpoint final é:

```bash
http://<load-balancer>.us-east-1.elb.amazonaws.com/v1/ai/turn
```

## Teste

```bash
curl -X POST http://<load-balancer>.us-east-1.elb.amazonaws.com/v1/ai/turn   -H "Content-Type: application/json"   -d '{}'
```


# 15. Troubleshooting Real Encontrado

## AWS CLI no WSL2 não encontrava `aws`
**Causa:** instalado apenas no Windows.  
**Solução:** instalar AWS CLI v2 no WSL2.

## Erro SSL no WSL2
**Causa:** proxy corporativo com inspeção TLS.  
**Solução:** instalar a CA corporativa no trust store do Linux.

## Erro SSL dentro do container
**Causa:** o container não confiava na CA corporativa.  
**Solução:** embutir `corp-root.crt` na imagem e rodar `update-ca-certificates`.

## Falha de autenticação AWS no container local
**Causa:** o container não via credenciais do host.  
**Solução:** montar `~/.aws` localmente ou passar variáveis temporárias.

## Pod em `Pending`
**Causa:** capacidade insuficiente de node para agendar o pod.  
**Solução:** usar o nodegroup final funcional.

## `504 Gateway Timeout`
**Causa:** Load Balancer existia, mas não havia pod saudável por trás.  
**Solução:** validar o pod antes de testar o endpoint.

## Exclusão do nodegroup travada
**Causa:** pod do CoreDNS impedia drain do único node.  
**Solução:** excluir o pod manualmente para destravar a remoção.


# 16. Checklist Operacional

- [ ] Validar ferramentas locais (`aws`, `docker`, `kubectl`, `eksctl`)
- [ ] Garantir confiança na CA corporativa no WSL2
- [ ] Garantir confiança na CA corporativa na imagem Docker
- [ ] Buildar a imagem local
- [ ] Testar localmente a API
- [ ] Criar repositório ECR
- [ ] Fazer login no ECR
- [ ] Taguear e enviar imagem
- [ ] Criar cluster EKS
- [ ] Validar node `Ready`
- [ ] Associar OIDC provider
- [ ] Criar policy IAM do Bedrock
- [ ] Criar ServiceAccount com IRSA
- [ ] Aplicar manifest Kubernetes
- [ ] Validar pod `Running`
- [ ] Validar `kubectl get svc`
- [ ] Testar o endpoint externo


# 17. Limpeza Completa do Ambiente

## Remover a aplicação

```bash
kubectl delete -f k8s/ems-mvp.yaml
```

## Remover nodegroup

```bash
eksctl delete nodegroup   --cluster ems-mvp-cluster   --region us-east-1   --name standard-workers-small
```

## Remover ServiceAccount com IRSA

```bash
eksctl delete iamserviceaccount   --cluster ems-mvp-cluster   --region us-east-1   --namespace default   --name ems-mvp-sa
```

## Remover cluster

```bash
eksctl delete cluster   --name ems-mvp-cluster   --region us-east-1
```

## Remover policy IAM

```bash
aws iam delete-policy   --policy-arn arn:aws:iam::746443183845:policy/BedrockAccessPolicy
```

## Remover repositório ECR

```bash
aws ecr delete-repository   --repository-name ems-mvp   --region us-east-1   --force
```

## Limpeza local do kubectl

```bash
kubectl config delete-context arn:aws:eks:us-east-1:746443183845:cluster/ems-mvp-cluster
kubectl config delete-cluster arn:aws:eks:us-east-1:746443183845:cluster/ems-mvp-cluster
kubectl config unset users.arn:aws:eks:us-east-1:746443183845:cluster/ems-mvp-cluster
```


# 18. Considerações de Produção

## O que esta arquitetura já faz certo

- usa container imutável
- usa ECR como registry corporativo
- usa Kubernetes gerenciado
- usa IAM sem access keys
- expõe aplicação por Load Balancer
- separa build, registry e runtime

## Melhorias recomendadas

1. adicionar readiness/liveness probes
2. externalizar variáveis em ConfigMap/Secret
3. adicionar observabilidade
4. automatizar com CI/CD
5. versionar manifests com Helm ou Kustomize
6. restringir a policy IAM por recursos específicos


# 19. Conclusão

Este notebook documenta a implantação completa e funcional da aplicação EMS MVP em Amazon EKS.

Ao final do processo, foi possível:

- empacotar a aplicação em Docker
- publicar a imagem no Amazon ECR
- criar e operar um cluster Amazon EKS
- configurar autenticação segura com IRSA
- implantar a aplicação em Kubernetes
- expor a API por AWS Load Balancer
- validar a chamada ponta a ponta até o Bedrock
- limpar totalmente o ambiente ao final

## Mensagem final para desenvolvedores

O valor desta arquitetura não está apenas em “fazer subir”.  
O valor está em compreender a responsabilidade de cada camada:

- **Docker** empacota
- **ECR** distribui
- **EKS** orquestra
- **IRSA** autentica
- **Service / LoadBalancer** expõem
- **Bedrock** entrega a capacidade de IA

Quando essa divisão fica clara, a implantação deixa de ser um conjunto de comandos e passa a ser uma arquitetura entendida e reproduzível.
